# 8.21 Semantic role labeling

Semantic role labeling turns a sentence from a string of words into a small event record: the predicate, its actors, its objects, and the circumstances around it.

SRL is the bridge between sequence tagging and meaning extraction. It anchors on a predicate, scores argument spans, and uses frame constraints to prevent locally plausible duplicate roles.

Save a copy to Drive to edit.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

def srl_ladder():
    return [
        {"name": "D1 lesson sell event", "predicate": 1, "tokens": ["Alice", "sold", "Bob", "a car"], "spans": [(0, 0), (2, 2), (3, 3)], "role_scores": np.array([[2.5, 0.4, 1.2], [0.2, 0.5, 2.1], [0.1, 2.4, 0.3]]), "gold": {((0, 0), "ARG0"), ((2, 2), "ARG2"), ((3, 3), "ARG1")}, "difficulty": 1},
        {"name": "D2 clean predicate arguments", "predicate": 2, "tokens": ["Mina", "gave", "Ravi", "notes"], "spans": [(0, 0), (2, 2), (3, 3)], "role_scores": np.array([[2.2, 0.3, 0.8], [0.4, 0.6, 2.0], [0.2, 2.1, 0.5]]), "gold": {((0, 0), "ARG0"), ((2, 2), "ARG2"), ((3, 3), "ARG1")}, "difficulty": 2},
        {"name": "D3 duplicate roles near ties", "predicate": 1, "tokens": ["The", "team", "sent", "files", "clients"], "spans": [(0, 1), (3, 3), (4, 4)], "role_scores": np.array([[2.0, 0.4, 0.4], [0.2, 2.1, 2.0], [0.2, 2.0, 2.2]]), "gold": {((0, 1), "ARG0"), ((3, 3), "ARG1"), ((4, 4), "ARG2")}, "difficulty": 4},
        {"name": "D4 inline PropBank snippets", "predicate": 3, "tokens": ["Yesterday", "Alice", "quickly", "lent", "Bob", "a laptop"], "spans": [(1, 1), (4, 4), (5, 5), (0, 0)], "role_scores": np.array([[2.3, 0.4, 0.5], [0.3, 0.7, 2.2], [0.2, 2.3, 0.7], [0.8, 0.3, 0.4]]), "gold": {((1, 1), "ARG0"), ((4, 4), "ARG2"), ((5, 5), "ARG1")}, "difficulty": 6},
        {"name": "D5 longer multi-predicate sentence", "predicate": 4, "tokens": ["After", "review", "Maya", "promised", "Liam", "to", "deliver", "the", "draft", "Friday"], "spans": [(2, 2), (4, 4), (7, 8), (9, 9), (0, 1)], "role_scores": np.array([[2.1, 0.5, 0.4], [0.3, 2.2, 2.1], [0.2, 2.0, 2.4], [0.8, 0.4, 0.3], [0.6, 0.4, 0.4]]), "gold": {((2, 2), "ARG0"), ((4, 4), "ARG2"), ((7, 8), "ARG1")}, "difficulty": 9},
    ]


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - arr.max()
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def role_f1(pred, gold):
    tp = len(pred & gold)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold) if gold else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


## The concept, built once (D1)\n\nFor predicate position $p$ and candidate span $s$, SRL scores frame-conditioned roles: $$P(r\mid s,p)=\operatorname{softmax}_r(h_s^\top W_rh_p+b_{f(p),r}),\qquad \hat r=\arg\max_rP(r\mid s,p)$$\n\nThe lesson `Alice sold Bob a car` has one predicate flag, ARG0/ARG1/ARG2 logits, duplicate-role constraints, and a 0.5 span margin.

In [ ]:
predicate_flags = [0, 1, 0, 0, 0]
assert sum(predicate_flags) == 1
probabilities = softmax([2.5, 0.4, 1.2])
assert round(float(probabilities[0]), 3) == 0.717
assert round(float(probabilities[1]), 3) == 0.088
assert round(float(probabilities[2]), 3) == 0.195
local_roles = ["ARG0", "ARG1", "ARG1"]
assert len(local_roles) == 3
assert len(set(local_roles)) == 2
assert len(local_roles) != len(set(local_roles))
scores = [0.1, 2.0, 0.5, 1.5]
assert max(scores) == 2.0
assert round(max(scores) - sorted(scores)[-2], 1) == 0.5
arg2_flags = [1, 1]
arg3_flags = [0, 1]
assert sum(arg2_flags) == 2
assert arg3_flags[1] > arg3_flags[0]
print("Lesson numbers verified")

Now build one reusable decoder. It turns role logits into probabilities, assigns at most one core role per predicate frame, and keeps span boundaries in the output facts.

In [ ]:
def srl_decode(predicate, spans, role_scores, constraints=True):
    roles = ["ARG0", "ARG1", "ARG2"]
    predictions = set()
    used = set()
    heat = []
    order = list(range(len(spans)))
    margins = role_scores.max(axis=1) - np.partition(role_scores, -2, axis=1)[:, -2]
    order.sort(key=lambda idx: (float(role_scores[idx].max()), float(margins[idx])), reverse=True)
    for idx in order:
        probs = softmax(role_scores[idx])
        role = roles[int(np.argmax(probs))]
        heat.append(probs)
        if constraints and role in used:
            alternatives = np.argsort(probs)[::-1]
            for alt in alternatives:
                candidate = roles[int(alt)]
                if candidate not in used:
                    role = candidate
                    break
        if role not in used:
            predictions.add((spans[idx], role))
            used.add(role)
    return predictions, np.array(heat)

rung = srl_ladder()[0]
pred, heat = srl_decode(rung["predicate"], rung["spans"], rung["role_scores"])
assert role_f1(pred, rung["gold"]) == 1.0
print(pred)

## The dataset ladder

The ladder starts with the sell-event toy, then adds clean predicates, duplicate-role near ties, inline PropBank-style snippets, and longer multi-predicate sentences.

In [ ]:
rungs = srl_ladder()
for rung in rungs:
    print(rung["name"], "tokens=", len(rung["tokens"]), "spans=", len(rung["spans"]))
    print("sample spans", rung["spans"][:3])

## Run the SAME method across D1-D5

In [ ]:
rows = []
for rung in rungs:
    pred, heat = srl_decode(rung["predicate"], rung["spans"], rung["role_scores"])
    score = role_f1(pred, rung["gold"])
    rows.append((rung["name"], rung["difficulty"], score, pred))
for name, difficulty, score, pred in rows:
    print(f"{name:34s} difficulty={difficulty:2d} F1={score:.3f} pred={sorted(pred)}")

## Results visualization

In [ ]:
fig, axes = plt.subplots(2, len(rungs), figsize=(16, 6))
metrics = []
difficulties = []
for col, rung in enumerate(rungs):
    pred, heat = srl_decode(rung["predicate"], rung["spans"], rung["role_scores"])
    metrics.append(role_f1(pred, rung["gold"]))
    difficulties.append(rung["difficulty"])
    axes[0, col].imshow(rung["role_scores"], cmap="Oranges")
    axes[0, col].set_title(rung["name"].split()[0])
    axes[0, col].set_xlabel("role")
    axes[0, col].set_ylabel("span")
    axes[1, col].bar(range(len(rung["spans"])), rung["role_scores"].max(axis=1))
    axes[1, col].set_title("span confidence")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(difficulties, metrics, marker="o")
plt.xlabel("sentence / argument complexity")
plt.ylabel("role-span F1")
plt.title("SRL F1 vs difficulty")
plt.ylim(0, 1.05)
plt.show()

## Pitfall on the hardest rung

Local argmax can assign duplicate ARG1 or decode against the wrong predicate. The fix uses predicate anchoring plus one-core-role frame constraints.

In [ ]:
hard = rungs[-1]
roles = ["ARG0", "ARG1", "ARG2"]
local = set()
for idx, span in enumerate(hard["spans"]):
    role = roles[int(np.argmax(hard["role_scores"][idx]))]
    local.add((span, role))
fixed, heat = srl_decode(hard["predicate"], hard["spans"], hard["role_scores"], constraints=True)
wrong_predicate = -1
if wrong_predicate != hard["predicate"]:
    wrong = set()
else:
    wrong = local
print("local argmax", local, "F1", role_f1(local, hard["gold"]))
print("wrong predicate", wrong, "F1", role_f1(wrong, hard["gold"]))
print("constrained", fixed, "F1", role_f1(fixed, hard["gold"]))
assert role_f1(fixed, hard["gold"]) >= role_f1(local, hard["gold"])
assert role_f1(wrong, hard["gold"]) == 0.0

## Evaluate it + Practice

- Metric: role-span F1.
- No-skill baseline: label each candidate span with its local argmax.
- Cheap sanity check: exact span boundaries appear in predicted facts.
- Ablation: remove frame uniqueness and duplicate core roles appear.
- Failure signals: missing predicate flags, duplicate ARG labels, or correct roles on wrong spans.

Practice 1: Change one D3 example so the pitfall becomes visible earlier, then recompute the metric.

Practice 2: Add one D5 example with a realistic edge case and explain whether the method should pass.

Practice 3: Turn off the key constraint and predict which rung loses the most metric.